## 1. Configuración de la ejecución
Los parámetros identifican las ingestas completas que ya validamos.

In [0]:
import json
import re
from functools import reduce
from delta.tables import DeltaTable
from pyspark.sql import functions as F

dbutils.widgets.text("run_procesos", "6717455f-416d-4457-9a88-600732d976d6")
dbutils.widgets.text("run_idartes", "95aa87aa-90ee-4104-8f38-d18edc3287b8")
dbutils.widgets.text("run_idipron", "90a6911a-40da-486b-b7c4-9ac9ab37ecfa")
dbutils.widgets.text("ambiente", "dev", "Ambiente: dev o prod")
dbutils.widgets.text("catalogo", "project_dev", "Catalogo de Unity Catalog")
dbutils.widgets.text("storage_account", "addb01", "Storage account del ambiente")

## 2. Lectura de Bronze y controles iniciales
Se consultan las tablas administradas de desarrollo, filtradas por entidad y Run ID. Antes de transformar se exige el conteo auditado: 
- 3651/3233 contratos y 3667/2888 filas de procesos.
- Los textos vacíos se convierten en `null`. 
- Los importes se tipan como `decimal`, sin adivinar separadores ni redondear valores invalidos.
- SQL se utiliza únicamente para inspeccionar metadatos y las transformaciones se implementan en PySpark.

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
def leer_uuid(nombre):
    valor = dbutils.widgets.get(nombre).strip()
    if not re.fullmatch(r"[0-9a-fA-F]{8}(?:-[0-9a-fA-F]{4}){3}-[0-9a-fA-F]{12}", valor):
        raise ValueError(f"UUID invalido: {nombre}")
    return valor

run_procesos = leer_uuid("run_procesos")
runs = {e: leer_uuid(f"run_{e}") for e in ("idartes", "idipron")}
ambiente = dbutils.widgets.get("ambiente").strip().lower()
catalogo = dbutils.widgets.get("catalogo").strip()
storage_account = dbutils.widgets.get("storage_account").strip()

if ambiente not in {"dev", "prod"}:
    raise ValueError("ambiente debe ser dev o prod")

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo):
    raise ValueError("Nombre de catalogo invalido")

if not re.fullmatch(r"[a-z0-9]{3,24}", storage_account):
    raise ValueError("Nombre de storage account invalido")

if ambiente == "prod" and (
    catalogo.lower() == "project_dev" or storage_account == "addb01"
):
    raise ValueError(
        "Para prod debes indicar su catalogo y storage account"
    )

esquema = f"{catalogo}.silver"
raiz_destino = f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"

print(f"Ambiente: {ambiente}")
print(f"Destino Silver: {esquema}")
print(f"Raiz de almacenamiento esperada: {raiz_destino}")

spark.sql(f"DESCRIBE SCHEMA EXTENDED {esquema}").collect()

contratos_b = spark.table(f"{catalogo}.bronze.secop_contratos").filter(
    ((F.col("_entidad") == "idartes") & (F.col("_run_origen") == runs["idartes"]))
    | ((F.col("_entidad") == "idipron") & (F.col("_run_origen") == runs["idipron"])))
procesos_b = spark.table(f"{catalogo}.bronze.secop_procesos").filter(
    F.col("_entidad").isin("idartes", "idipron") & (F.col("_run_origen") == run_procesos))

def comprobar_conteos(df, columna, esperados, nombre):
    observados = {r[columna]: r['count'] for r in df.groupBy(columna).count().collect()}
    if observados != esperados:
        raise ValueError(f"{nombre}: conteos inesperados: {observados}; esperados {esperados}")

comprobar_conteos(contratos_b, "_entidad", {"idartes": 3651, "idipron": 3233}, "Bronze contratos")
comprobar_conteos(procesos_b, "_entidad", {"idartes": 3667, "idipron": 2888}, "Bronze procesos")

def texto(df, nombre):
    valor = F.trim(F.col(nombre).cast("string")) if nombre in df.columns else F.lit(None).cast("string")
    return F.when(valor == "", F.lit(None).cast("string")).otherwise(valor)

def decimal_seguro(columna):
    # No adivinar separadores ni redondear silenciosamente cantidades con mas de dos decimales.
    return F.when(columna.rlike(r"^[+-]?\d{1,36}(?:\.\d{1,2})?$"), columna.cast("decimal(38,2)"))

Ambiente: dev
Destino Silver: project_dev.silver
Raiz de almacenamiento esperada: abfss://lakehouse@addb01.dfs.core.windows.net/


## 3. Contratos: tipado, trazabilidad y calidad
- Se conserva una fila por entidad, contrato y snapshot. 
- Una clave duplicada detiene la carga en lugar de escoger una fila arbitraria.
- Las fechas se convierten a `date` y el valor contractual a `decimal`. 
- Se conservan los valores originales para revisar errores de conversión.
- Los indicadores qc_* señalan problemas, no eliminan contratos. 
- No se incluyen cuentas bancarias ni datos personales de representantes o supervisores.
- La trazabilidad conserva el archivo Raw, el Run ID y el hash de contenido registrados en Bronze.

In [0]:
# Contratos: una fila por entidad/contrato/run de origen; no deduplicar silenciosamente.
contratos = contratos_b.select(
    F.col("_entidad").alias("entidad"),
    texto(contratos_b, "id_contrato").alias("id_contrato"),
    texto(contratos_b, "proceso_de_compra").alias("id_portafolio"),
    texto(contratos_b, "nombre_entidad").alias("nombre_entidad"),
    texto(contratos_b, "nit_entidad").alias("nit_entidad"),
    texto(contratos_b, "referencia_del_contrato").alias("referencia_contrato"),
    texto(contratos_b, "estado_contrato").alias("estado_contrato"),
    texto(contratos_b, "tipo_de_contrato").alias("tipo_contrato"),
    texto(contratos_b, "modalidad_de_contratacion").alias("modalidad_contratacion"),
    texto(contratos_b, "objeto_del_contrato").alias("objeto_contrato"),
    texto(contratos_b, "proveedor_adjudicado").alias("proveedor_adjudicado"),
    texto(contratos_b, "documento_proveedor").alias("documento_proveedor"),
    texto(contratos_b, "tipodocproveedor").alias("tipo_documento_proveedor"),
    texto(contratos_b, "codigo_proveedor").alias("codigo_proveedor"),
    texto(contratos_b, "fecha_de_firma").alias("fecha_firma_original"),
    texto(contratos_b, "fecha_de_inicio_del_contrato").alias("fecha_inicio_original"),
    texto(contratos_b, "fecha_de_fin_del_contrato").alias("fecha_fin_original"),
    texto(contratos_b, "valor_del_contrato").alias("valor_contrato_original"),
    "_run_origen", "_archivo_origen", "_contenido_sha256", "_anio_cohorte_contratos")

for original, destino in (("fecha_firma_original", "fecha_firma"),
                           ("fecha_inicio_original", "fecha_inicio"),
                           ("fecha_fin_original", "fecha_fin")):
    contratos = contratos.withColumn(destino, F.to_date(F.try_to_timestamp(F.col(original))))
contratos = (contratos.withColumn("valor_contrato", decimal_seguro(F.col("valor_contrato_original")))
             .withColumn("qc_valor_invalido", F.col("valor_contrato_original").isNull() | F.col("valor_contrato").isNull())
             .withColumn("qc_fecha_firma_invalida", F.col("fecha_firma").isNull())
             .withColumn("qc_fecha_inicio_invalida", F.col("fecha_inicio_original").isNotNull() & F.col("fecha_inicio").isNull())
             .withColumn("qc_fecha_fin_invalida", F.col("fecha_fin_original").isNotNull() & F.col("fecha_fin").isNull())
             .withColumn("qc_valor_negativo", F.coalesce(F.col("valor_contrato") < 0, F.lit(False))))
if contratos.filter(F.col("id_contrato").isNull() | F.col("id_portafolio").isNull()).limit(1).count():
    raise ValueError("Contratos con claves nulas")
if contratos.groupBy("entidad", "_run_origen", "id_contrato").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Clave contrato duplicada: revisar; no escoger una fila arbitrariamente")

## 4. Procesos: consolidar repeticiones sin perder variantes
- Se agrupan exclusivamente filas con el mismo contenido normalizado, entidad y Run ID.
- Un mismo `id_del_proceso` puede tener varios contenidos: esas variantes se conservan, sin decidir arbitrariamente cuál es la correcta.
- `filas_bronze_agrupadas` y `archivos_origen` permiten rastrear las repeticiones consolidadas.

Resultado esperado: **IDARTES** pasa de 3667 a 3630 filas (37 repeticiones) e **IDIPRON** conserva 2888. Total Silver: 6518 variantes.
Estos conteos no son el número de IDs de proceso distintos, ambas medidas se muestran por separado.

In [0]:
# Agrupar por contenido preserva variantes con el mismo id_del_proceso.
# El struct de datos es identico dentro de cada hash; no se elige una variante por fecha.
campos_origen = sorted(c for c in procesos_b.columns if not c.startswith("_"))
procesos_unicos = procesos_b.groupBy("_entidad", "_run_origen", "_contenido_sha256").agg(
    F.first(F.struct(*[F.col(c) for c in campos_origen])).alias("dato"),
    F.count(F.lit(1)).alias("filas_bronze_agrupadas"),
    F.sort_array(F.collect_set("_archivo_origen")).alias("archivos_origen"))
procesos_unicos = procesos_unicos.select("dato.*", "_entidad", "_run_origen", "_contenido_sha256",
                                       "filas_bronze_agrupadas", "archivos_origen")
procesos = procesos_unicos.select(
    F.col("_entidad").alias("entidad"),
    texto(procesos_unicos, "id_del_proceso").alias("id_proceso"),
    texto(procesos_unicos, "id_del_portafolio").alias("id_portafolio"),
    texto(procesos_unicos, "referencia_del_proceso").alias("referencia_proceso"),
    texto(procesos_unicos, "modalidad_de_contratacion").alias("modalidad_contratacion"),
    texto(procesos_unicos, "fase").alias("fase"),
    texto(procesos_unicos, "estado_del_procedimiento").alias("estado_procedimiento"),
    texto(procesos_unicos, "estado_resumen").alias("estado_resumen"),
    texto(procesos_unicos, "adjudicado").alias("adjudicado_original"),
    texto(procesos_unicos, "id_adjudicacion").alias("id_adjudicacion"),
    texto(procesos_unicos, "precio_base").alias("precio_base_original"),
    texto(procesos_unicos, "valor_total_adjudicacion").alias("valor_adjudicacion_original"),
    texto(procesos_unicos, "fecha_de_publicacion_del").alias("fecha_publicacion_original"),
    "_run_origen", "_contenido_sha256", "filas_bronze_agrupadas", "archivos_origen")
procesos = (procesos.withColumn("precio_base", decimal_seguro(F.col("precio_base_original")))
            .withColumn("valor_adjudicacion", decimal_seguro(F.col("valor_adjudicacion_original")))
            .withColumn("fecha_publicacion", F.to_date(F.try_to_timestamp("fecha_publicacion_original")))
            .withColumn("qc_precio_base_invalido", F.col("precio_base_original").isNotNull() & F.col("precio_base").isNull())
            .withColumn("qc_valor_adjudicacion_invalido", F.col("valor_adjudicacion_original").isNotNull() & F.col("valor_adjudicacion").isNull())
            .withColumn("qc_fecha_publicacion_invalida", F.col("fecha_publicacion_original").isNotNull() & F.col("fecha_publicacion").isNull()))
if procesos.filter(F.col("id_proceso").isNull() | F.col("id_portafolio").isNull()).limit(1).count():
    raise ValueError("Procesos con claves nulas")
comprobar_conteos(procesos, "entidad", {"idartes": 3630, "idipron": 2888}, "Silver procesos por contenido")

## 5. Portafolios y cruce sin multiplicar contratos
- La relación correcta es `contratos.proceso_de_compra = procesos.id_del_portafolio`.
- Se construye una fila por entidad y portafolio con la lista de IDs y sus conteos. Esa tabla evita un join directo contrato-proceso de uno a muchos.
- Un left join conserva todos los contratos: tiene_proceso=false identifica las no coincidencias, sin calificarlas como irregularidades.

Controles esperados: 6884 contratos despues del cruce; 3589 coincidencias IDARTES y 2860 IDIPRON. Se conservan 435 contratos sin coincidencia.
Las tablas siguientes muestran problemas de conversion y repeticiones. Revisa esos resultados antes de construir indicadores Gold.

In [0]:
# Una sola fila por portafolio/entidad/snapshot: nunca sumar contratos tras un join 1:N.
portafolios = procesos.groupBy("entidad", "id_portafolio").agg(
    F.count(F.lit(1)).alias("variantes_proceso"),
    F.countDistinct("id_proceso").alias("numero_procesos_distintos"),
    F.sort_array(F.collect_set("id_proceso")).alias("ids_procesos"),
    F.sort_array(F.collect_set("modalidad_contratacion")).alias("modalidades_procesos"),
    F.sum("filas_bronze_agrupadas").alias("filas_bronze_procesos"))
portafolios = portafolios.withColumn("_run_procesos", F.lit(run_procesos))
contratos = (contratos.join(portafolios.drop("_run_procesos"), ["entidad", "id_portafolio"], "left")
             .withColumn("tiene_proceso", F.col("numero_procesos_distintos").isNotNull())
             .withColumn("numero_procesos_distintos", F.coalesce("numero_procesos_distintos", F.lit(0)))
             .withColumn("variantes_proceso", F.coalesce("variantes_proceso", F.lit(0)))
             .withColumn("_run_procesos", F.lit(run_procesos)))
comprobar_conteos(contratos, "entidad", {"idartes": 3651, "idipron": 3233}, "Silver contratos despues del cruce")
comprobar_conteos(contratos.filter(F.col("tiene_proceso")), "entidad", {"idartes": 3589, "idipron": 2860}, "Cobertura")

calidad = contratos.groupBy("entidad").agg(
    F.count(F.lit(1)).alias("contratos"),
    F.sum(F.col("tiene_proceso").cast("long")).alias("con_proceso"),
    F.sum((~F.col("tiene_proceso")).cast("long")).alias("sin_proceso"),
    F.sum(F.col("qc_valor_invalido").cast("long")).alias("valores_invalidos"),
    F.sum(F.col("qc_fecha_firma_invalida").cast("long")).alias("fechas_firma_invalidas"),
    F.sum(F.col("qc_fecha_inicio_invalida").cast("long")).alias("fechas_inicio_invalidas"),
    F.sum(F.col("qc_fecha_fin_invalida").cast("long")).alias("fechas_fin_invalidas"),
    F.sum(F.col("qc_valor_negativo").cast("long")).alias("valores_negativos"))
display(calidad.orderBy("entidad"))
display(procesos.groupBy("entidad").agg(
    F.count(F.lit(1)).alias("variantes_silver"),
    F.countDistinct("id_proceso").alias("ids_distintos"),
    F.sum(F.col("filas_bronze_agrupadas") - 1).alias("duplicados_consolidados"),
    F.sum(F.col("qc_precio_base_invalido").cast("long")).alias("precios_invalidos"),
    F.sum(F.col("qc_valor_adjudicacion_invalido").cast("long")).alias("adjudicaciones_invalidas"),
    F.sum(F.col("qc_fecha_publicacion_invalida").cast("long")).alias("fechas_publicacion_invalidas")
).orderBy("entidad"))

entidad,contratos,con_proceso,sin_proceso,valores_invalidos,fechas_firma_invalidas,fechas_inicio_invalidas,fechas_fin_invalidas,valores_negativos
idartes,3651,3589,62,0,0,0,0,0
idipron,3233,2860,373,0,0,0,0,0


entidad,variantes_silver,ids_distintos,duplicados_consolidados,precios_invalidos,adjudicaciones_invalidas,fechas_publicacion_invalidas
idartes,3630,3629,37,0,0,0
idipron,2888,2885,0,0,0,0


## 6. Persistencia de Silver en Delta
- Se escriben `secop_contratos`, `secop_procesos` y `secop_portafolios` en `project_dev.silver`.
- Son tablas administradas: Unity Catalog resuelve su almacenamiento en `addb01/lakehouse`. No se escribe en Raw ni se sobrescribe Bronze.
- Si la tabla ya existe, un MERGE por claves del snapshot actualiza o inserta esas filas. No borra otros snapshots ni incrementa filas al repetir la misma carga.
- Se verifica la unicidad de claves, la ubicacion y los conteos persistidos. No ejecutes dos copias simultaneamente.

Resumen esperado: 6884 contratos, 6518 variantes de procesos y 6447 portafolios encontrados.

In [0]:
def guardar_silver(df, nombre, claves):
    tabla = f"{esquema}.{nombre}"
    if df.groupBy(*claves).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{tabla}: clave Silver no unica")
    if spark.catalog.tableExists(tabla):
        detalle = spark.sql(f"DESCRIBE DETAIL {tabla}").first().asDict()
        descripcion = spark.sql(f"DESCRIBE TABLE EXTENDED {tabla}").collect()
        tipo = next((r.data_type for r in descripcion if r.col_name.strip() == "Type"), None)
        if tipo != "MANAGED" or detalle["format"] != "delta" or not detalle["location"].startswith(raiz_destino):
            raise ValueError(f"{tabla}: destino no es Delta administrado en lakehouse de desarrollo")
        if set(df.columns) != set(spark.table(tabla).columns):
            raise ValueError(f"{tabla}: esquema previo incompatible; no sobrescribir")
        condicion = reduce(lambda a, b: a & b,
                           [F.col(f"t.{c}") == F.col(f"s.{c}") for c in claves])
        (DeltaTable.forName(spark, tabla).alias("t").merge(df.alias("s"), condicion)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    else:
        df.write.format("delta").mode("append").saveAsTable(tabla)
    detalle = spark.sql(f"DESCRIBE DETAIL {tabla}").first().asDict()
    if not detalle["location"].startswith(raiz_destino):
        raise ValueError(f"{tabla}: se creo fuera del lakehouse esperado: {detalle['location']}")
    # Comparacion multiconjunto de las claves de esta carga; no contar otros snapshots.
    guardadas = spark.table(tabla).join(df.select(*claves), claves, "left_semi")
    if guardadas.count() != df.count():
        raise ValueError(f"{tabla}: conteo persistido incorrecto")
    return {"tabla": tabla, "filas_snapshot": df.count(), "ubicacion_delta": detalle["location"]}

salida = []
salida.append(guardar_silver(procesos, "secop_procesos",
                           ["entidad", "_run_origen", "_contenido_sha256"]))
salida.append(guardar_silver(portafolios, "secop_portafolios",
                           ["entidad", "_run_procesos", "id_portafolio"]))
salida.append(guardar_silver(contratos, "secop_contratos",
                           ["entidad", "_run_origen", "_run_procesos", "id_contrato"]))
display(spark.createDataFrame(salida))
print("Silver creado. Contratos sin proceso conservados y variantes de procesos no descartadas.")

filas_snapshot,tabla,ubicacion_delta
6518,project_dev.silver.secop_procesos,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/368f0cb9-d050-4b0c-94d3-2c4aac4ce55f
6447,project_dev.silver.secop_portafolios,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/f2afef5b-8cb2-4523-b245-d8ba6937c50b
6884,project_dev.silver.secop_contratos,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/45696ce0-a982-42db-94fe-4e039f553c7a


Silver creado. Contratos sin proceso conservados y variantes de procesos no descartadas.


## 7. Resultado para orquestación
Este JSON devuelve un resumen con tablas, conteos y ubicaciones. Una actividad Notebook de ADF puede consumirlo en runOutput.
Esta salida solo aparece al superar los controles anteriores.

In [0]:
dbutils.notebook.exit(json.dumps({"tablas": salida, "run_procesos": run_procesos}, ensure_ascii=False))